# Tweet Preprocessing and Sentiment Labelling

This notebook processes tweets from the tweets folder, cleans them, and applies FinRoBERTa sentiment analysis.

**Process:**
1. Load tweet CSV files from tweets directory
2. Clean and preprocess tweet text (handles mentions, hashtags, URLs, emojis)
3. Apply FinRoBERTa sentiment analysis
4. Save labelled tweets with sentiment scores

## 1. Import Required Libraries

Import libraries for data processing, sentiment analysis, and file handling.

In [1]:
import os
import re
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from langdetect import detect, detect_langs, LangDetectException, DetectorFactory
DetectorFactory.seed = 0
from tqdm import tqdm

## 2. Directories and Settings

Set input directory (raw tweets) and output directory (processed tweets with sentiment).

**Directory Structure:**
- Input: `tweets/` folder containing `tweets_TICKER.csv` files
- Output: `tweets_labelled/` folder for processed results

In [2]:
# Directories definitions
BASE_DIR = Path.cwd().resolve()
if (BASE_DIR / "tweets").exists():
    pass
elif (BASE_DIR.parent / "tweets").exists():
    BASE_DIR = BASE_DIR.parent

INPUT_DIR = str(BASE_DIR / "tweets")
OUTPUT_DIR = BASE_DIR / "tweets_labelled"

print(f"BASE_DIR: {BASE_DIR}")
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_DIR: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP
INPUT_DIR: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\tweets
OUTPUT_DIR: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\tweets_labelled


## 3. Load FIN-RoBERTa Model

FIN-RoBERTa is a custom RoBERTa model fine-tuned on financial text for sentiment analysis.

**Model Details:**
- Model: alasteirho/FIN-RoBERTa-Custom
- Output: 3 classes (positive, negative, neutral)
- Sentiment Score: P(positive) - P(negative), range [-1, 1]

In [ ]:
# Load FinRoBERTa tokenizer and model for financial sentiment analysis
tokenizer = AutoTokenizer.from_pretrained('alasteirho/FIN-RoBERTa-Custom')
model = AutoModelForSequenceClassification.from_pretrained('alasteirho/FIN-RoBERTa-Custom')

# Set to evaluation mode (disables dropout)
model.eval()

# Use GPU if available for faster inference
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
model.to(device)


print(f"Using device: {device}")

Using device: cuda


## 4. Define Tweet Preprocessing Function

Clean tweets for FinRoBERTa sentiment analysis.

**Cleaning Steps:**
- Remove URLs and shortened links
- Handle Twitter mentions (@username) - keep or remove based on context
- Process hashtags (convert #CamelCase to readable text)
- Remove excessive punctuation and emojis
- Filter out very short tweets (< 10 characters)
- **Filter out non-English tweets using langdetect**

In [4]:
def is_english(text, min_prob=0.4, min_alpha_chars=20, min_word_hits=2):
    """Check if text is English using langdetect with normalization."""
    if not text or not isinstance(text, str):
        return False

    # REGEX definition: Non beneficial characters that affect language detection.
    sample = re.sub(r'http\S+|www\.\S+', ' ', text)
    sample = re.sub(r'@\w+', ' ', sample)
    sample = re.sub(r'\$[A-Z]{1,5}\b', ' ', sample)
    sample = sample.replace('#', ' ')
    sample = re.sub(r'[^A-Za-z\s]', ' ', sample)
    sample = re.sub(r'\s+', ' ', sample).strip()

    if not sample:
        return False

    tokens = sample.lower().split()
    alpha_chars = len(re.sub(r'[^A-Za-z]', '', sample))
    if alpha_chars < min_alpha_chars:
        # Error handling to avoid false negatives on shorter texts
        return True

    english_hints = {
        'the', 'and', 'is', 'to', 'of', 'for', 'in', 'on', 'with', 'as',
        'at', 'from', 'this', 'that', 'it', 'be', 'are', 'was', 'will',
        'good', 'bad', 'buy', 'sell', 'long', 'short', 'risk', 'setup',
        'target', 'entry', 'stop', 'loss', 'support', 'resistance',
        'break', 'trend', 'chart', 'price', 'market', 'level', 'move',
        'hold', 'watch', 'news', 'update', 'downgrade', 'upgrade',
    }

    hint_hits = sum(1 for word in tokens if word in english_hints)
    if hint_hits >= min_word_hits:
        return True

    try:
        langs = detect_langs(sample)
    except LangDetectException:
        return False

    for lang in langs:
        if lang.lang == 'en' and lang.prob >= min_prob:
            return True

    # If English the top detected language but slightly under threshold, keep it.
    if langs and langs[0].lang == 'en':
        return True

    return False

def preprocess_tweet(tweet_text):
    # Handle missing or invalid tweets
    if pd.isna(tweet_text) or not isinstance(tweet_text, str):
        return None
    
    # Remove newlines and carriage returns - make everything single line
    tweet_text = tweet_text.replace('\n', ' ').replace('\r', ' ')
    
    # Normalize whitespace
    tweet_text = re.sub(r'\s+', ' ', tweet_text).strip()
    
    # Remove URLs and shortened links (http, https, bit.ly, etc.)
    tweet_text = re.sub(r'http\S+|www\.\S+', '', tweet_text)
    tweet_text = re.sub(r'bit\.ly/\S+|goo\.gl/\S+', '', tweet_text)
    
    # Remove Twitter mentions at the start of tweets (replies)
    tweet_text = re.sub(r'^(@\w+\s*)+', '', tweet_text)
    
    # Converts hashtags to readable text by adding spaces before capitals
    # Example: #AppleStock -> Apple Stock
    def hashtag_to_text(match):
        tag = match.group(1)
        # Add space before capital letters
        spaced = re.sub(r'([a-z])([A-Z])', r'\1 \2', tag)
        return spaced
    
    tweet_text = re.sub(r'#(\w+)', hashtag_to_text, tweet_text)
    
    # Remove stock ticker symbols like $AAPL, $TSLA as they do not provide any sentiment value
    tweet_text = re.sub(r'\$[A-Z]{1,5}\b', '', tweet_text)
    
    # Remove excessive punctuation (>3 repeated characters)
    tweet_text = re.sub(r'([!?.])\\1{2,}', r'\1', tweet_text)
    
    # Remove emojis and special characters
    tweet_text = re.sub(r'[^\w\s.,!?-]', '', tweet_text)
    
    # Remove extra whitespace again after all cleaning
    tweet_text = re.sub(r'\s+', ' ', tweet_text).strip()
    
    # Filter out very short tweets (likely not meaningful, too short for sentiment analysis)
    if len(tweet_text) < 10:
        return None
    
    # Filter out non-English tweets, the model is trained on English data only
    if not is_english(tweet_text):
        return None
    
    return tweet_text

## 5. Define Sentiment Analysis Function

Apply FinRoBERTa model to get sentiment scores for cleaned tweets.

**Sentiment Calculation:**
- Gets probability distribution over [negative, neutral, positive]
- Sentiment score = P(positive) - P(negative)
- Range: -1 (very negative) to +1 (very positive)

In [5]:
def get_sentiment(text):
    # Handle empty or None text
    if not text or pd.isna(text):
        return None
    
    # Tokenize and prepare input for FinRoBERTa
    inputs = tokenizer(
        text, 
        return_tensors='pt', 
        truncation=True, 
        max_length=512,
        padding=True
    )
    
    # Move input tensors to same device as model
    inputs = {key: value.to(device) for key, value in inputs.items()}
    
    # Get model predictions without computing gradients
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    # Extract probabilities: [negative, neutral, positive]
    probabilities = predictions[0].cpu().numpy()
    positive_prob = probabilities[2]
    negative_prob = probabilities[0]
    
    # Calculate sentiment score: positive - negative
    sentiment_score = positive_prob - negative_prob
    
    return sentiment_score

## 6. Get List of Tweet CSV Files

Find all tweet CSV files in the input directory (format: `tweets_TICKER.csv`).

In [6]:
# Get all CSV files from tweets directory
csv_files = [
    filename 
    for filename in os.listdir(INPUT_DIR) 
    if filename.endswith('.csv') and filename.startswith('tweets_')
]

print(f"Found {len(csv_files)} tweet files to process:")
for file in csv_files:
    print(f"  - {file}")

# Load all data for analysis
all_dfs = {}
for csv_file in csv_files:
    ticker = csv_file.replace('tweets_', '').replace('.csv', '')
    all_dfs[ticker] = pd.read_csv(os.path.join(INPUT_DIR, csv_file))

Found 17 tweet files to process:
  - tweets_AAPL.csv
  - tweets_AMD.csv
  - tweets_AMZN.csv
  - tweets_AVGO.csv
  - tweets_BRK.B.csv
  - tweets_GOOGL.csv
  - tweets_HD.csv
  - tweets_JPM.csv
  - tweets_LLY.csv
  - tweets_MA.csv
  - tweets_META.csv
  - tweets_MSFT.csv
  - tweets_NVDA.csv
  - tweets_ORCL.csv
  - tweets_TSLA.csv
  - tweets_V.csv
  - tweets_XOM.csv


## 7. Data Quality Analysis

Before processing, analyze the raw data for:
- NA/missing values statistics
- Daily tweet coverage (including weekends)
- Non-English tweet examples

### 7.1 NA Values Statistics

In [7]:
for ticker, df in all_dfs.items():
    print(f"\n--- {ticker} ({len(df)} total tweets) ---")
    
    # Count NA values per column
    na_counts = df.isna().sum()
    na_percentages = (df.isna().sum() / len(df) * 100).round(2)
    
    na_stats = pd.DataFrame({
        'NA Count': na_counts,
        'NA %': na_percentages
    })
    
    print("\nNA values per column:")
    display(na_stats[na_stats['NA Count'] > 0] if na_stats['NA Count'].sum() > 0 else "No NA values found!")
    
    # Show sample rows with NA in body column
    if df['body'].isna().sum() > 0:
        print(f"\nSample rows with NA in 'body' column:")
        display(df[df['body'].isna()].head(5))


--- AAPL (23530 total tweets) ---

NA values per column:


,NA Count,NA %
body,5,0.02
post_date,79,0.34
likes,426,1.81



Sample rows with NA in 'body' column:


,ticker,search_date,username,body,post_date,replies,retweets,likes
23105,AAPL,2025-10-10,"Grow your stack with Mamo. \n\nDeposit, earn p...",NaN,44,124,1200,NaN
23191,AAPL,2025-10-10,AI SDK Agents,NaN,26,12,388,NaN
23225,AAPL,2025-10-10,AI SDK Agents,NaN,26,15,391,NaN
23342,AAPL,2025-10-10,"Grow your stack with Mamo. \n\nDeposit, earn p...",NaN,58,238,1800,NaN
23495,AAPL,2025-10-10,"Grow your stack with Mamo. \n\nDeposit, earn p...",NaN,66,246,1800,NaN



--- AMD (20178 total tweets) ---

NA values per column:


...

,ticker,search_date,username,body,post_date,replies,retweets,likes
1279,V,2023-12-11,V.T,NaN,2023-12-11T23:59:28.000Z,0,0,7.0
1297,V,2023-12-11,V-Ness,NaN,2023-12-11T23:59:29.000Z,0,0,1.0
2012,V,2024-01-22,Vladimir,NaN,2024-01-22T23:59:15.000Z,0,0,2.0
2125,V,2024-01-23,VAS,NaN,2024-01-23T23:59:37.000Z,0,0,0.0
2171,V,2024-01-23,Dr. Erin Wilson,NaN,2024-01-23T23:59:49.000Z,0,0,1.0



--- XOM (34413 total tweets) ---

NA values per column:


,NA Count,NA %
username,27,0.08
body,47,0.14
post_date,24,0.07
likes,25034,72.75



Sample rows with NA in 'body' column:


,ticker,search_date,username,body,post_date,replies,retweets,likes
378,XOM,2023-10-22,ぷぷ♪,NaN,2023-10-22T00:40:23.000Z,1,0,11.0
3337,XOM,2024-02-24,jane #TheFutureIsFresh,NaN,2024-02-24T07:57:32.000Z,1,2,13.0
5789,XOM,2024-06-22,ぷぷ♪,NaN,2024-06-22T10:00:27.000Z,0,0,25.0
6223,XOM,2024-07-13,jane #TheFutureIsFresh,NaN,2024-07-13T11:00:50.000Z,0,0,0.0
6855,XOM,2024-08-06,Crimson King,NaN,2024-08-06T22:26:27.000Z,0,0,0.0


### 7.2 Daily Tweet Coverage (Including Weekends)

In [8]:
for ticker, df in all_dfs.items():
    print(f"\n--- {ticker} ---")
    
    # Parse date column (post_date is the actual tweet date)
    df['date'] = pd.to_datetime(df['post_date']).dt.date
    
    # Count tweets per day
    daily_counts = df.groupby('date').size().reset_index(name='tweet_count')
    daily_counts['date'] = pd.to_datetime(daily_counts['date'])
    daily_counts['day_of_week'] = daily_counts['date'].dt.day_name()
    
    # Get date range
    min_date = daily_counts['date'].min()
    max_date = daily_counts['date'].max()
    
    # Create complete date range to find missing days
    full_date_range = pd.date_range(start=min_date, end=max_date, freq='D')
    missing_dates = set(full_date_range) - set(daily_counts['date'])
    
    print(f"\nDate range: {min_date.date()} to {max_date.date()}")
    print(f"Total days in range: {len(full_date_range)}")
    print(f"Days with tweets: {len(daily_counts)}")
    print(f"Missing days: {len(missing_dates)}")
    
    # Show tweets by day of week
    dow_counts = daily_counts.groupby('day_of_week')['tweet_count'].agg(['sum', 'mean', 'count'])
    dow_counts = dow_counts.reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
    dow_counts.columns = ['Total Tweets', 'Avg Tweets/Day', 'Days Count']
    print("\nTweets by Day of Week:")
    display(dow_counts)
    
    # Show if any days are missing
    if missing_dates:
        print(f"\nMissing dates (first 10):")
        for d in sorted(list(missing_dates))[:10]:
            print(f"  {d.date()} ({d.day_name()})")
    else:
        print("\n All days have tweets (no gaps)")
    
    # Show sample of daily counts
    print("\nSample daily tweet counts:")
    display(daily_counts[['date', 'day_of_week', 'tweet_count']].head(10))


--- AAPL ---


ValueError: time data "0" doesn't match format "%Y-%m-%dT%H:%M:%S.%f%z", at position 23104. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

### 7.3 Non-English Tweets Detection and Examples

In [ ]:
def detect_language(text):
    """Detect language of text, return language code or 'unknown'."""
    if pd.isna(text) or not isinstance(text, str) or len(text.strip()) < 10:
        return 'too_short'
    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'

for ticker, df in all_dfs.items():
    print(f"\n--- {ticker} ---")
    
    # Detect language for each tweet
    print("Detecting languages (this may take a moment)...")
    df['detected_lang'] = df['body'].apply(detect_language)
    
    # Language statistics
    lang_counts = df['detected_lang'].value_counts()
    lang_percentages = (lang_counts / len(df) * 100).round(2)
    
    lang_stats = pd.DataFrame({
        'Count': lang_counts,
        'Percentage': lang_percentages
    })
    
    print("\nLanguage Distribution:")
    display(lang_stats.head(15))
    
    # Count non-English tweets
    english_count = lang_counts.get('en', 0)
    non_english_count = len(df) - english_count - lang_counts.get('too_short', 0) - lang_counts.get('unknown', 0)
    
    print(f"\nSummary:")
    print(f"  English tweets: {english_count} ({english_count/len(df)*100:.1f}%)")
    print(f"  Non-English tweets: {non_english_count} ({non_english_count/len(df)*100:.1f}%)")
    print(f"  Too short to detect: {lang_counts.get('too_short', 0)}")
    print(f"  Unknown/Error: {lang_counts.get('unknown', 0)}")
    
    # Show examples of non-English tweets
    non_english_df = df[(df['detected_lang'] != 'en') & 
                        (df['detected_lang'] != 'too_short') & 
                        (df['detected_lang'] != 'unknown')]
    
    if len(non_english_df) > 0:
        print(f"\n--- EXAMPLES OF NON-ENGLISH TWEETS (will be removed) ---")
        # Group by language and show examples
        for lang in non_english_df['detected_lang'].unique()[:5]:
            lang_examples = non_english_df[non_english_df['detected_lang'] == lang].head(3)
            print(f"\n[{lang.upper()}] - {len(non_english_df[non_english_df['detected_lang'] == lang])} tweets:")
            for idx, row in lang_examples.iterrows():
                body_preview = row['body'][:150] + "..." if len(str(row['body'])) > 150 else row['body']
                print(f"  - {body_preview}")
        
        # Display as DataFrame
        print(f"\nNon-English tweets sample (DataFrame view):")
        display(non_english_df[['body', 'detected_lang']].head(10))
    else:
        print("\nAll tweets are in English!")


--- AAPL ---
Detecting languages (this may take a moment)...

Language Distribution:


,Count,Percentage
detected_lang,,
en,20569,88.87
ja,428,1.85
too_short,338,1.46
es,303,1.31
ca,186,0.80
de,134,0.58
ar,112,0.48
af,102,0.44
fr,102,0.44



Summary:
  English tweets: 20569 (88.9%)
  Non-English tweets: 2236 (9.7%)
  Too short to detect: 338
  Unknown/Error: 1

--- EXAMPLES OF NON-ENGLISH TWEETS (will be removed) ---

[AF] - 102 tweets:
  - top llm developers by vaulation 

$msft $tsla $aapl $goog $meta $nvda
  - Not a good look. $AAPL
  - $AAPL not a good look boss

[ES] - 303 tweets:
  - Hay más de 10 empresas proveedoras de chips detrás de un Apple 15 pro. Fantástica infografía realizada por 
@Quartr_App
 $AAPL
  - Principales proveedores de chips para el iPhone 15 Pro de $AAPL , la mayoría de los cuales cotizan en bolsa.
  - Y si hermano, si 
@cocoscap
 es la única empresa que te permite comprar 1 CEDEAR de $AAPL por $10.000 sin pagar ni un solo peso en comisiones!

[CA] - 186 tweets:
  - $AAPL - 15 min  via  
alerttrade.us 
dlvr.it/SxCbnj
  - $AAPL: 180 Not Bad? 
dlvr.it/SxMQlG  via  
alerttrade.us
  - $AAPL 30%+

[ID] - 24 tweets:
  - $AAPL bear pennant
  - Si me dan a elegir una $AAPL, si me dan dos $AAPL y $MSFT.


,body,detected_lang
13,top llm developers by vaulation \n\n$msft $tsl...,af
19,Hay más de 10 empresas proveedoras de chips de...,es
33,Principales proveedores de chips para el iPhon...,es
36,$AAPL - 15 min via \nalerttrade.us \ndlvr.it...,ca
39,"Y si hermano, si \n@cocoscap\n es la única emp...",es
44,$AAPL bear pennant,id
67,La #ComisiónEuropea busca información para det...,es
68,muy guapa la manzana $AAPL en su canal\n\n#ICE...,es
70,$AAPL good setup 178 is risk,hr
136,米国主要企業の売上構成比を比較\n\n10月は米国大型株の決算が続きます\nアップル( $A...,ja



--- AMD ---
Detecting languages (this may take a moment)...

Language Distribution:


,Count,Percentage
detected_lang,,
en,18858,87.02
too_short,541,2.50
ja,340,1.57
de,230,1.06
es,226,1.04
ar,126,0.58
af,112,0.52
tr,107,0.49
tl,93,0.43



Summary:
  English tweets: 18858 (87.0%)
  Non-English tweets: 2270 (10.5%)
  Too short to detect: 541
  Unknown/Error: 3

--- EXAMPLES OF NON-ENGLISH TWEETS (will be removed) ---

[TL] - 93 tweets:
  - $AMD Easy.
  - $AMD ongoing breakout
  - $AMD Flagging intraday

[DE] - 230 tweets:
  - AI SDK Agents
  - $NVDA +1,3% über 470$
$SMH heute stärkster Sektor +1,3%
$AMD ebenso +2,35% up
  - $AMD 10/27 $102 PUTS @ 1.80 - 1.85

FULL SWING 

[HR] - 5 tweets:
  - $AMD take me to 109.4
  - WATCHLIST

$AMD
$CELH
$U
$PYPL
$PATH
$ZM
  - $tsla $googl $amd $smci

[ES] - 226 tweets:
  - $AMD este setup esta muy caliente
  - $AMD liberando precio.
  - Advanced Micro Devices  gráfico semanal: semáforo  oscilador  soporte $76 resistencia $126 $AMD

[IT] - 68 tweets:
  - Level to level $AMD
  - AMD $AMD Intel $INTC
  - $AMD compression

Non-English tweets sample (DataFrame view):


,body,detected_lang
6,$AMD Easy.,tl
13,AI SDK Agents,de
32,$AMD take me to 109.4,hr
40,$AMD este setup esta muy caliente,es
52,$AMD ongoing breakout,tl
55,Level to level $AMD,it
60,$amd 109 on deck,sv
66,"$AMD oznámila, že plánuje koupit startup zabýv...",cs
88,さすがにご祝儀買いは入らないか $AMD,ja
107,$AMD liberando precio.,es


## 8. Process All Tweet Files

For each ticker's tweet file:
1. Load tweets from CSV
2. Clean tweet text using preprocessing function
3. Filter out non-English and invalid tweets
4. Apply FinRoBERTa sentiment analysis
5. Save results with sentiment scores

**Output Format:**
- All original columns preserved
- New column: `cleaned_body` (preprocessed tweet text)
- New column: `sentiment` (FinRoBERTa sentiment score)

In [ ]:
# Process each tweet file
for csv_file in csv_files:
    print(f"\n{'='*60}")
    print(f"Processing {csv_file}...")
    print(f"{'='*60}")
    
    # Step 1: Load tweet data
    input_path = os.path.join(INPUT_DIR, csv_file)
    df = pd.read_csv(input_path)
    
    print(f"\n--- STEP 1: Raw Data Loaded ({len(df)} tweets) ---")
    display(df.head(5))
    
    # Step 2: Clean tweet text
    print(f"\n--- STEP 2: Cleaning tweets... ---")
    df['cleaned_body'] = df['body'].apply(preprocess_tweet)
    
    print("After cleaning (showing body vs cleaned_body):")
    display(df[['body', 'cleaned_body']].head(10))
    
    # Step 3: Remove rows with invalid/empty tweets after cleaning
    initial_count = len(df)
    df = df.dropna(subset=['cleaned_body'])
    removed_count = initial_count - len(df)
    
    print(f"\n--- STEP 3: Removed non-English/invalid tweets ---")
    print(f"Removed {removed_count} tweets, {len(df)} remaining")
    display(df.head(5))
    
    # Step 4: Apply sentiment analysis
    print(f"\n--- STEP 4: Analyzing sentiment... ---")
    tqdm.pandas(desc="Progress")
    df['sentiment'] = df['cleaned_body'].progress_apply(get_sentiment)
    
    # Remove any rows where sentiment analysis failed
    df = df.dropna(subset=['sentiment'])
    
    print("\nAfter sentiment analysis:")
    display(df[['cleaned_body', 'sentiment']].head(10))
    
    # Step 5: Keep only essential columns (date, cleaned text for display, sentiment)
    print(f"\n--- STEP 5: Preparing final dataframe ---")
    df['date'] = pd.to_datetime(df['post_date']).dt.date
    df = df[['date', 'cleaned_body', 'sentiment']]
    df = df.rename(columns={'cleaned_body': 'headline'})
    
    print("Dataframe preview (headline shown for reference only):")
    display(df.head(10))
    
    # Step 6: Save processed data (only date and sentiment, not headline)
    output_path = os.path.join(OUTPUT_DIR, csv_file)
    df_to_save = df[['date', 'sentiment']]
    df_to_save.to_csv(output_path, index=False)
    
    print(f"\n--- STEP 6: Final CSV Saved ({len(df_to_save)} tweets) ---")
    print(f"Saved columns: {list(df_to_save.columns)}")
    display(df_to_save.head(5))
    
    # Display statistics
    print(f"\nSentiment statistics:")
    print(f"  Mean: {df['sentiment'].mean():.4f}")
    print(f"  Median: {df['sentiment'].median():.4f}")
    print(f"  Std Dev: {df['sentiment'].std():.4f}")
    print(f"\nSaved to: {output_path}")

## 9. Summary

All tweet files have been processed successfully!

**Output CSV Format (2 columns):**
- `date`: Tweet date (extracted from post_date)
- `sentiment`: FinRoBERTa sentiment score [-1, 1]

**Note:** The `headline` (cleaned tweet text) is displayed in the notebook for reference but NOT saved to the CSV.

**Dropped columns:** ticker, search_date, username, body, post_date, replies, retweets, likes, headline

**Next Steps:**
- Check `tweets_labelled/` folder for output files
- These files can now be used for aggregation and ML pipeline integration